In [ ]:
import json
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "numind/NuExtract3"

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()

def run_nuextract(messages, **chat_template_kwargs):
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        **chat_template_kwargs,
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=False,
        )

    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()
df = pd.DataFrame(columns = ["image", "texte"])


In [ ]:
image_folder = "" #le chemin relatif vers le dossier contenant les images
output_csv = "" #le nom souhaité pour le fichier de sortie (ex : "output.csv") 


In [ ]:
from pathlib import Path
for i in Path.iterdir(Path.cwd()/image_folder):
  image = Image.open(i).convert("RGB")
  image.thumbnail((1024, 1024))
  receipt_messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "image",
                  "image": image,
              }
          ],
      "system": '''Transcribe the zone in the red square and ONLY this one in the red_squared_text field'''}# à remplacer par le prompt de son choix selon les besoins 
  ]

  template = {"red_squared_text":"verbatim-string"
            } #à remplacer par le schéma d'output de son choix. Consulter la doc de nu-extract 3 pour voir comment structurer son schéma. 


  structured_output = run_nuextract(
      receipt_messages,
      template=json.dumps(template, indent=4),
      enable_thinking=False,
      temperature = 0.1
  )
  print(structured_output)
  dico = {"image":i, "texte":json.loads(structured_output)["red_squared_text"]}

  print(dico)
  df.loc[len(df)] = dico

In [ ]:
df.to_csv(output_csv)